# FEM Plate-with-Hole GNN Surrogate — Kaggle GPU setup

Sibling notebook to `colab_setup.ipynb`, adapted for Kaggle instead of Colab.
Much simpler than the AirfRANS project's Kaggle notebook: that one had to
stream-extract a ~15GB external dataset one case at a time to fit Kaggle's
disk quota. This dataset (200 Abaqus cases, ~207MB raw + ~78MB cached) is
small enough to already be committed to the repo -- clone and go, no
download/streaming step needed.

No Drive-equivalent live mount here: Kaggle persists across sessions via
**Datasets** (read-only, attached to a session) and a notebook's own
**Output** (via "Save Version"), not a synced folder -- checkpoints below go
to `/kaggle/working/`, which becomes that notebook's Output when you save a
version.

Before running: Settings (right sidebar) > Accelerator > GPU, and
Internet > On (needed for git clone / pip install).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the repo

Pushed to `https://github.com/Revanthkr1/fem-plate-gnn` (**private**) --
`data/raw/*.json` (200 cases) and `data/norm_stats.npz` are committed, so they
arrive with the clone; only `data/cache/` (gitignored, rebuilt in section 2)
is missing after this.

Because the repo is private, plain `git clone` would prompt for credentials
and fail non-interactively. The cell below reads a GitHub personal access
token from Kaggle's **Secrets** add-on instead of hardcoding one -- add a
token there first: notebook menu > Add-ons > Secrets, named `GITHUB_TOKEN`
(generate one, classic, `repo` scope, at
https://github.com/settings/tokens). Never paste a real token directly into
this notebook -- it gets committed to git.

(Alternative: make the repo public later once you're ready to share it, and
plain `git clone` works with no token at all.)

In [ ]:
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{token}@github.com/Revanthkr1/fem-plate-gnn.git"

!git clone $REPO_URL repo
%cd repo

## 2. Install dependencies + rebuild the cache

Kaggle ships torch with CUDA already installed -- don't reinstall it.
`torch_geometric` installs as pure Python (same `torch_geometric.utils.scatter`-only
usage as AirfRANS's model.py, ported unchanged). No `airfrans` package needed
-- this project doesn't depend on it.

`data/cache/` (preprocessed graph tensors) is gitignored and cheap to rebuild
-- these are small meshes (~5-6k nodes, not AirfRANS's ~180k), so this is
seconds. Safe to re-run: `preprocess_case` skips any case already cached.

In [ ]:
!pip install -q torch_geometric lightning pyvista pyyaml

In [ ]:
import glob
import os

from src.preprocess import preprocess_split

RAW_DIR = "data/raw"
CACHE_DIR = "/kaggle/working/cache"

n_cases = len(glob.glob(os.path.join(RAW_DIR, "case_*.json")))
case_ids = list(range(n_cases))
preprocess_split(RAW_DIR, case_ids, CACHE_DIR)
print(f"cached {len(glob.glob(os.path.join(CACHE_DIR, 'case_*.pt')))}/{n_cases} cases")

## 3. Smoke test: one case on the real GPU (optional)

Same forward/backward timing check as the Colab notebook's section 4 --
confirms the model + a real cached graph actually move to the GPU and run
before committing to a full training loop. Skip if you're just resuming a
training run.

In [ ]:
import time
import numpy as np
import torch

from src.dataset import CachedPyGPlateHoleDataset
from src.model import MeshGraphNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

stats = dict(np.load("data/norm_stats.npz"))
ds = CachedPyGPlateHoleDataset(CACHE_DIR, [0], stats=stats)
data = ds[0].to(device)

model = MeshGraphNet(node_in_dim=4, edge_in_dim=2, out_dim=3).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

t0 = time.time()
pred = model(data.x, data.edge_index, data.edge_attr)
loss = torch.nn.functional.mse_loss(pred, data.y)
loss.backward()
opt.step()
print(f"nodes={data.x.shape[0]}, edges={data.edge_index.shape[1]}, "
      f"forward+backward+step={time.time()-t0:.2f}s, loss={loss.item():.4f}")

## 4. Train

Trains on 200 cases minus a 20-case validation holdout (`n_val=20`, ~10% --
see `configs/base.yaml`), reporting relative L2 per field (`u_x`, `u_y`,
`von_mises` separately -- never one blended number, per CLAUDE.md) on
validation each epoch.

**This run starts fresh, deliberately -- not resuming.** The first 100-epoch
run's peak-stress accuracy was weak (37% relative error, 18mm location miss
on the holdout -- see `PROJECT_FLOW.md` phase 8); `max_epochs` is now 250 to
test whether more training alone helps, as a clean single-variable
experiment. `/kaggle/working/` starts empty each fresh session regardless, so
this mostly matters if you reattach a previous run's saved Output as an input
dataset -- checkpoint path below is `meshgraphnet_250ep.ckpt`, distinct from
the first run's `meshgraphnet.ckpt`, so `resume_from_checkpoint` can't
auto-detect and resume into a checkpoint whose Cosine annealing schedule was
built for the old `max_epochs=100` (risks a corrupted LR schedule, and would
muddy whether any improvement came from this change or from carryover).

Checkpoints go to `/kaggle/working/` -- click **Save Version** (periodically,
or once done) to persist them past this session, same as the AirfRANS
project's Kaggle notebook. If this session disconnects before you save a
version, the run is lost and needs restarting -- unlike Colab+Drive, there's
no live-synced equivalent here.

These graphs are much smaller than AirfRANS's, so OOM headroom is not the
concern `batch_size=1`/`accumulate_grad_batches=4` addressed there -- kept the
same defaults anyway since they're harmless at this scale and a larger batch
size hasn't been benchmarked yet.

In [ ]:
import glob
import os
import yaml

from src.train import main as train_main

config = yaml.safe_load(open("configs/base.yaml"))
n_cases = len(glob.glob(os.path.join(RAW_DIR, "case_*.json")))
CHECKPOINT_PATH = "/kaggle/working/meshgraphnet_250ep.ckpt"  # new path -- see section 4 markdown for why

train_main(
    cache_dir=CACHE_DIR,
    stats_path="data/norm_stats.npz",
    checkpoint_path=CHECKPOINT_PATH,
    case_ids=list(range(n_cases)),
    model_kwargs=config["model"],
    max_epochs=config["training"]["max_epochs"],
    batch_size=config["training"]["batch_size"],
    accumulate_grad_batches=config["training"]["accumulate_grad_batches"],
    n_val=config["training"]["n_val"],
    lr=config["training"]["lr"],
    checkpoint_every_n_epochs=config["training"]["checkpoint_every_n_epochs"],
    num_workers=config["training"]["num_workers"],
    precision="16-mixed",  # GPU-specific override -- base.yaml's 32-true is for local CPU runs
)